# Reconstruct experiment stimuli

Requires local MATLAB files, excluded from Git. Run from the repository root. Labels follow stimulus conditions (nint), not participant answers (resp). Filenames are numbered within condition rather than by original trial ID. Existing WAVs are never overwritten.

In [ ]:
import numpy as np
import scipy.io as sio
import soundfile as sf
from pathlib import Path

if not Path('cfg/SincNet_Exp.cfg').is_file():
    raise RuntimeError('Run this notebook from the repository root.')
output_dir = Path('exp_data')
output_dir.mkdir(parents=True, exist_ok=True)
if any(output_dir.glob('*.wav')):
    raise FileExistsError('exp_data already contains WAVs; keep them or archive them before regenerating.')

# Load MATLAB data
gr2data = sio.loadmat('gr2data.mat')
stim_params_gr2 = sio.loadmat('stim_params_gr2.mat')

# Extract the necessary fields from the loaded data
data = gr2data['data']
orderdb = stim_params_gr2['orderdb']
freqval = stim_params_gr2['freqval'].flatten()  # Flattening to ensure it's 1D
orderphase = stim_params_gr2['orderphase']

# Extract the 'nint' field from each track within the data
nnint = []
for kk in range(data.shape[1]):  # Iterate through the trials
    nnint.extend(data[0, kk]['track'][0, 0]['nint'][0, 0].flatten())

# Prepare the parameters
nnint = np.array(nnint) - 1
rampdur = 0.005
fs = 44100
dur = 0.2
t = np.arange(0, dur, 1 / fs)
ntrials = len(nnint)
if not np.isin(nnint, [0, 1]).all():
    raise ValueError('Expected nint values 1 and 2 before conversion.')
if freqval.size != 23 or orderdb.shape[1] != 23 or orderphase.shape[1] != 23:
    raise ValueError('Expected 23 acoustic components.')
if orderdb.shape[0] < ntrials or orderphase.shape[0] < ntrials:
    raise ValueError('Missing acoustic parameters for some trials.')

# Convert dB to amplitude
orderamps = 10 ** (orderdb / 20)
orderamps = orderamps / 500000  # Adjust as needed

# Function to apply ramp damping
def rampdamp(signal, rampdur, fs):
    ramp_len = int(rampdur * fs)
    ramp = np.linspace(0, 1, ramp_len)
    signal[:ramp_len] *= ramp
    signal[-ramp_len:] *= ramp[::-1]
    return signal

# Ensure the signal length is not smaller than the window length
wlen = len(t)
sigstim = np.zeros(len(t))
nosigstim = np.zeros(len(t))

# Generate signal with stimulus (for nnint == 1)
idxs = np.where(nnint == 1)[0]
for itrial in range(len(idxs)):
    sigstim.fill(0)  # Reset the signal for each trial
    for jj in range(23):
        # Ensure the dimensions match for broadcasting
        freq = freqval[jj]
        phase = orderphase[idxs[itrial], jj]
        amp = orderamps[idxs[itrial], jj]
        
        # Ensure the freqval and phase arrays match the dimensions of t
        sigstim += amp * np.cos(2 * np.pi * freq * t + phase)
    
    sigstim = rampdamp(sigstim, rampdur, fs)
    wvfile = f'exp_data/sigwf{itrial + 1}.wav'
    sf.write(wvfile, sigstim, fs)

# Generate no-signal stimulus (for nnint == 0)
idxns = np.where(nnint == 0)[0]
for itrial in range(len(idxns)):
    nosigstim.fill(0)  # Reset the signal for each trial
    for jj in range(23):
        freq = freqval[jj]
        phase = orderphase[idxns[itrial], jj]
        amp = orderamps[idxns[itrial], jj]
        
        nosigstim += amp * np.cos(2 * np.pi * freq * t + phase)
    
    nosigstim = rampdamp(nosigstim, rampdur, fs)
    wvfile = f'exp_data/nswf{itrial + 1}.wav'
    sf.write(wvfile, nosigstim, fs)
